In [1]:
import sys
sys.path.append("/kaggle/input/datasets/yusufhilal/full-model/Event-Extraction-Model")

In [2]:
import numpy as np
import pandas as pd
import torch
import random


from sklearn.model_selection import GroupShuffleSplit, KFold
from transformers import AutoTokenizer

from helpers.utils import load_config, compute_num_stats
from helpers.dataset import merge_labels
from helpers.losses import bio_loss
from helpers.metrics import find_best_threshold_peak, boundary_metrics_peak
from helpers.train_utils import make_loader, run_epoch
from models.dom_extractor import init_model_and_optim, set_bert_trainable

In [3]:
# config --------------------------------------------------------------
cfg = load_config()

cfg["model"]["d_model"] = 128
cfg["model"]["nhead"] = 4
cfg["model"]["num_layers"] = 2
cfg["model"]["dropout"] = 0.3
cfg["model"]["max_tokens"] = 32

cfg["training"]["freeze_epochs"] = 5
cfg["training"]["batch_size"] = 1 # fixed
cfg["training"]["n_splits"] = 1 # 1 for testing, 3 for improvements, 5 for final
cfg["training"]["seed"] = 5

cfg["features"]["num_cols"] = []
# cfg["features"]["bool_cols"] = []

In [4]:
SEED = cfg["training"]["seed"]
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [5]:
# load data and preprocess -----------------------------------------------
df = pd.read_csv("/kaggle/input/datasets/yusufhilal/full-model/Event-Extraction-Model/data/full_data.csv")

df = merge_labels(df, "label")
df = df.sort_values(["source", "rendering_order"]).reset_index(drop=True) # extra cleaninig

df["in_event"] = df["event_id"].notna().astype(int) # binary event flag column
df["start_event"] = 0

m = df["event_id"].notna() # mask for events only

# take index of first row of each event (event boundary)
first_idx = df.loc[m].groupby(["source", "event_id"], sort=False).head(1).index 
df.loc[first_idx, "start_event"] = 1
df["start_event"] = df["start_event"].fillna(0).astype(int)

# outside, begin, inside tagging
df["bio"] = 0 # outside
df.loc[df["in_event"].eq(1), "bio"] = 2 # events inside
df.loc[df["start_event"].eq(1), "bio"] = 1 # boundary -> beginning

print(f"Pages: {df['source'].nunique()}")
print(f"Total nodes: {len(df)}")
print(f"Label counts:\n{df['label'].value_counts()}")

Pages: 15
Total nodes: 3100
Label counts:
label
Other          2456
Date            242
Location        139
Name            120
Time            116
Description      27
Name: count, dtype: int64


In [6]:
# Params ---------------------------------------------------------------
# sort labels, make them integers for training and return mapping to labels for eval
LABELS = sorted(df["label"].unique().tolist())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

TAG_VOCAB = {t: i for i, t in enumerate(sorted(df["tag"].astype(str).unique().tolist()))}
PARENT_TAG_VOCAB = {t: i for i, t in enumerate(sorted(df["parent_tag"].astype(str).unique().tolist()))}

NUM_COLS = cfg["features"]["num_cols"]
BOOL_COLS = cfg["features"]["bool_cols"]

print(f"Labels: {LABELS}")
print(f"Tag vocab size: {len(TAG_VOCAB)}")
print(f"Parent tag vocab size: {len(PARENT_TAG_VOCAB)}")
print(f"Num features: {len(NUM_COLS)}")
print(f"Bool features: {len(BOOL_COLS)}")

Labels: ['Date', 'Description', 'Location', 'Name', 'Other', 'Time']
Tag vocab size: 15
Parent tag vocab size: 19
Num features: 0
Bool features: 20


In [7]:
# Train/Test Split ------------------------------------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.11, random_state=cfg["training"]["seed"])
train_idx, test_idx = next(gss.split(df, groups=df["source"]))

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

cv_sources = train_df["source"].unique()
test_sources = test_df["source"].unique()

print(f"\nTrain pages: {len(cv_sources)}")
print(f"Test pages: {len(test_sources)}")


Train pages: 13
Test pages: 2


In [8]:
# Tokenizer -------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["name"])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
# Cross Validation -----------------------------------------------------
N_SPLITS = min(cfg["training"]["n_splits"], len(cv_sources))
cv_results = []

if cfg["training"]["n_splits"] == 1:
    # single fold - just do a quick train/val split
    gss_cv = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=cfg["training"]["seed"])
    splits = list(gss_cv.split(cv_sources, groups=cv_sources))
    kf_splits = [(splits[0][0], splits[0][1])]
else:
    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=cfg["training"]["seed"])
    kf_splits = list(kf.split(cv_sources))
    

for fold, (tr_idx, va_idx) in enumerate(kf_splits, start=1):
        
    torch.cuda.empty_cache()
    
    fold_train_sources = set(cv_sources[tr_idx])
    fold_val_sources = set(cv_sources[va_idx])

    fold_train_df = df[df["source"].isin(fold_train_sources)].copy()
    fold_val_df = df[df["source"].isin(fold_val_sources)].copy()

    num_mean, num_std = compute_num_stats(fold_train_df, NUM_COLS)
    loss_fn = bio_loss(DEVICE)
    
    print(f"\n===== Fold {fold}/{N_SPLITS} =====")
    print(f"Train pages: {fold_train_df['source'].nunique()} Val pages: {fold_val_df['source'].nunique()}")

    train_loader = make_loader(
        fold_train_df, tokenizer,
        TAG_VOCAB, PARENT_TAG_VOCAB,
        NUM_COLS, BOOL_COLS,
        num_mean, num_std,
        batch_size=cfg["training"]["batch_size"],
        max_tokens=cfg["model"]["max_tokens"],
        shuffle=True
    )
    
    val_loader = make_loader(
        fold_val_df, tokenizer,
        TAG_VOCAB, PARENT_TAG_VOCAB,
        NUM_COLS, BOOL_COLS,
        num_mean, num_std,
        batch_size=cfg["training"]["batch_size"],
        max_tokens=cfg["model"]["max_tokens"],
        shuffle=False
    )

    model, optimizer = init_model_and_optim(
        cfg,
        tag_vocab_size=len(TAG_VOCAB),
        parent_tag_vocab_size=len(PARENT_TAG_VOCAB),
        num_numeric_features=len(NUM_COLS),
        num_bool_features=len(BOOL_COLS),
        device=DEVICE
    )
    
    set_bert_trainable(model, False)
    best = {"f1": -1.0, "th": 0.5, "state": None}

    for epoch in range(cfg["training"]["epochs"]):

        if epoch == cfg["training"]["freeze_epochs"]:
            set_bert_trainable(model, True)

        tr_loss = run_epoch(model, optimizer, train_loader, loss_fn, DEVICE, training=True)
        val_loss = run_epoch(model, optimizer, val_loader, loss_fn, DEVICE, training=False)

        th, f1 = find_best_threshold_peak( val_loader, model, DEVICE,
                                          nms_k=cfg["inference"]["nms_k"],
                                          min_gap=cfg["inference"]["min_gap"],
                                          tol=cfg["inference"]["tol"]
                                          )

        if f1 > best["f1"]:
            best["f1"] = f1
            best["th"] = th
            best["state"] ={k: v.cpu() for k, v in model.state_dict().items()}
            
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d} | tr_loss={tr_loss:.4f} val_loss={val_loss:.4f} best_f1={best['f1']:.4f} best_th={best['th']:.2f}")


    model.load_state_dict(best["state"])

    bp, br, bf1 = boundary_metrics_peak(
        val_loader, model, DEVICE,
        threshold=best["th"],
        nms_k=cfg["inference"]["nms_k"],
        min_gap=cfg["inference"]["min_gap"],
        tol=cfg["inference"]["tol"]
    )
    
    print(f"Fold {fold} | P={bp:.4f} R={br:.4f} F1={bf1:.4f} (th={best['th']:.2f})")

    cv_results.append({
        "bf1": bf1,
        "th": best["th"],
    })

    del model, optimizer
    torch.cuda.empty_cache()

# CV summary
mean_bf1 = float(np.mean([x["bf1"] for x in cv_results]))
mean_th = float(np.mean([x["th"] for x in cv_results]))
best_th_cv = mean_th

print(f"\n===== CV Summary =====")
print(f"Mean F1: {mean_bf1:.4f}")
print(f"Mean threshold: {mean_th:.4f}")
print(f"Using threshold for final eval: {best_th_cv:.4f}")


===== Fold 1/1 =====
Train pages: 10 Val pages: 3


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/nn/init.py:566: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:515: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, a

Epoch 05 | tr_loss=0.5880 val_loss=0.6378 best_f1=0.0870 best_th=0.05
Epoch 10 | tr_loss=0.2933 val_loss=0.5596 best_f1=0.0870 best_th=0.05
Epoch 15 | tr_loss=0.1375 val_loss=0.6280 best_f1=0.1633 best_th=0.10
Epoch 20 | tr_loss=0.0767 val_loss=0.7154 best_f1=0.1905 best_th=0.10
Fold 1 | P=0.6667 R=0.1111 F1=0.1905 (th=0.10)

===== CV Summary =====
Mean F1: 0.1905
Mean threshold: 0.1000
Using threshold for final eval: 0.1000


In [13]:
# Final train on all data -----------------------------------------------------
print("\nTraining final model on all CV data...")
final_train_df = df[df["source"].isin(set(cv_sources))].copy()
final_num_mean, final_num_std = compute_num_stats(final_train_df, NUM_COLS)

train_loader = make_loader(
    final_train_df, tokenizer,
    TAG_VOCAB, PARENT_TAG_VOCAB,
    NUM_COLS, BOOL_COLS,
    final_num_mean, final_num_std,
    batch_size=cfg["training"]["batch_size"],
    max_tokens=cfg["model"]["max_tokens"],
    shuffle=True
)

loss_fn = bio_loss(DEVICE)

model, optimizer = init_model_and_optim(
    cfg,
    tag_vocab_size=len(TAG_VOCAB),
    parent_tag_vocab_size=len(PARENT_TAG_VOCAB),
    num_numeric_features=len(NUM_COLS),
    num_bool_features=len(BOOL_COLS),
    device=DEVICE
)

set_bert_trainable(model, False)

for epoch in range(cfg["training"]["epochs"]):
    if epoch == cfg["training"]["freeze_epochs"]:
        set_bert_trainable(model, True)

    tr_loss = run_epoch(model, optimizer, train_loader, loss_fn, DEVICE, training=True)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | tr_loss={tr_loss:.4f}")

print("Final model training done.")


Training final model on all CV data...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/torch/nn/init.py:566: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


Epoch 05 | tr_loss=0.5781
Epoch 10 | tr_loss=0.2487
Epoch 15 | tr_loss=0.1165
Epoch 20 | tr_loss=0.0527
Final model training done.


In [14]:
# Holdout evaluation ------------------------------------------------------------
test_loader = make_loader(
    test_df, tokenizer,
    TAG_VOCAB, PARENT_TAG_VOCAB,
    NUM_COLS, BOOL_COLS,
    final_num_mean, final_num_std,
    batch_size=cfg["training"]["batch_size"],
    max_tokens=cfg["model"]["max_tokens"],
    shuffle=False
)

p, r, f1 = boundary_metrics_peak(
    test_loader, model, DEVICE,
    threshold=best_th_cv,
    nms_k=cfg["inference"]["nms_k"],
    min_gap=cfg["inference"]["min_gap"],
    tol=cfg["inference"]["tol"]
)

print(f"\n===== HOLDOUT TEST =====")
print(f"START: P={p:.4f} R={r:.4f} F1={f1:.4f}")
print(f"Threshold used: {best_th_cv:.4f}")


===== HOLDOUT TEST =====
START: P=0.2500 R=0.1667 F1=0.2000
Threshold used: 0.1000


In [15]:
# ── PER PAGE BOUNDARY VERIFICATION ────────────────────────────────────────
from helpers.metrics import collect_page_probs_and_truth, pick_starts_from_probs, start_prf_with_tolerance

pages = collect_page_probs_and_truth(test_loader, model, DEVICE)

for i, (prob_B, true_start) in enumerate(pages):
    source = test_sources[i]
    page_df = test_df[test_df["source"] == source].sort_values("rendering_order").reset_index(drop=True)

    true_starts = np.where(true_start == 1)[0].tolist()
    pred_starts = pick_starts_from_probs(
        prob_B,
        threshold=0.10,   # use debug threshold
        nms_k=cfg["inference"]["nms_k"],
        min_gap=cfg["inference"]["min_gap"]
    )

    p, r, f1 = start_prf_with_tolerance(true_starts, pred_starts, tol=cfg["inference"]["tol"])

    print(f"\n========== Page: {source} ==========")
    print(f"Nodes: {len(page_df)}")
    print(f"True starts:  {true_starts}")
    print(f"Pred starts:  {pred_starts}")
    print(f"P={p:.4f} R={r:.4f} F1={f1:.4f}")

    print(f"\n--- Probability window around each true start ---")
    for idx in true_starts:
        lo = max(0, idx - 2)
        hi = min(len(prob_B), idx + 3)
        print(f"\nWindow around true start {idx}:")
        for j in range(lo, hi):
            print(f"  Index {j:3d} | Prob_B={prob_B[j]:.3f}")

    print(f"\n--- Full node table ---")
    for j, row in page_df.iterrows():
        marker = ""
        if j in true_starts: marker += " [TRUE]"
        if j in pred_starts: marker += " [PRED]"
        txt = str(row.get("text_context", ""))[:50]
        print(f"  {j:3d} | ProbB={prob_B[j]:.3f} | {txt}{marker}")


========== Page: gpacac.net_pattern_labeled ==========
Nodes: 109
True starts:  [75, 77, 79, 83, 85]
Pred starts:  [73, 80, 106]
P=1.0000 R=2.0000 F1=4.0000

--- Probability window around each true start ---

Window around true start 75:
  Index  73 | Prob_B=0.113
  Index  74 | Prob_B=0.004
  Index  75 | Prob_B=0.091
  Index  76 | Prob_B=0.038
  Index  77 | Prob_B=0.036

Window around true start 77:
  Index  75 | Prob_B=0.091
  Index  76 | Prob_B=0.038
  Index  77 | Prob_B=0.036
  Index  78 | Prob_B=0.015
  Index  79 | Prob_B=0.101

Window around true start 79:
  Index  77 | Prob_B=0.036
  Index  78 | Prob_B=0.015
  Index  79 | Prob_B=0.101
  Index  80 | Prob_B=0.604
  Index  81 | Prob_B=0.001

Window around true start 83:
  Index  81 | Prob_B=0.001
  Index  82 | Prob_B=0.004
  Index  83 | Prob_B=0.078
  Index  84 | Prob_B=0.033
  Index  85 | Prob_B=0.020

Window around true start 85:
  Index  83 | Prob_B=0.078
  Index  84 | Prob_B=0.033
  Index  85 | Prob_B=0.020
  Index  86 | Prob_B

In [ ]:
# Save checkpoint ----------------------------------------------------------------
checkpoint = {
    # model weights
    "model_state_dict": model.state_dict(),

    # vocabs needed to rebuild the model and dataset
    "label2id": label2id,
    "id2label": id2label,
    "tag_vocab": TAG_VOCAB,
    "parent_tag_vocab": PARENT_TAG_VOCAB,

    # normalization stats needed for inference
    "num_mean": final_num_mean,
    "num_std": final_num_std,

    # feature columns so inference knows what to expect
    "num_cols": NUM_COLS,
    "bool_cols": BOOL_COLS,

    # best threshold from CV
    "best_th": best_th_cv,

    # config used for this run
    "cfg": cfg,
}

torch.save(checkpoint, "models/dom_extractor_checkpoint_v1.pt")
print("Checkpoint saved to models/dom_extractor_checkpoint.pt")